In [109]:
import pandas as pd
import os
import re
import numpy as np

In [110]:
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input/whirlpool")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output/whirlpool")

In [111]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    df = df.copy()

    # Extrae el sector: lo que está entre TX: y el siguiente :
    # "Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ" -> "AGRC"
    df["sector"] = df[strategy_col].str.extract(r"TX:([A-Z]{3,6}):", expand=False)

    # Caso especial baseline
    df.loc[df[strategy_col].str.contains(r"TX:BASE", regex=True, na=False), "sector"] = "BASE"

    # Extrae transformation_name: lo que está después de TX:SECTOR:
    # "Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ" -> "DEC_CH4_RICE_STRATEGY_NZ from NZ"
    df["transformation_name"] = df[strategy_col].str.extract(r"TX:[A-Z]{3,6}:(\S+)", expand=False)

    # Si quieres solo hasta el espacio (sin "from NZ"), usa \S+ que ya captura hasta el primer espacio
    # Si quieres todo lo que sigue incluyendo "from NZ", cambia \S+ por (.+)

    # Caso baseline
    base_mask = df[strategy_col].str.contains(r"TX:BASE", regex=True, na=False)
    df.loc[base_mask, "transformation_name"] = "BASE"

    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

## Load and process emission data

In [112]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "raw_emissions_uganda_2019_whirlpool_data_raw.csv"))
# emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_bulgaria_2022_trww_debug.csv"))
emissions_df.head()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
0,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.241921,2019,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
1,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.244762,2020,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
2,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252282,2021,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
3,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.251119,2022,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
4,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252083,2023,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE


In [113]:
print(emissions_df.primary_id.nunique())

61


In [114]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE', 'HBLE',
       'Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ from HBLE',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_NZ from HBLE',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_NZ from HBLE',
       'Remove TX:FRST:INCREASE_SEQUESTRATION_NZ from HBLE',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ from HBLE',
       'Remove TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_HFCS_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_N2O_STRATEGY_NZ from HBLE',
       'Remove TX:IPP

In [115]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

62
60


In [116]:
filtered_emissions_df["strategy"].unique()

array(['HBLE', 'Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ from HBLE',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_NZ from HBLE',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_NZ from HBLE',
       'Remove TX:FRST:INCREASE_SEQUESTRATION_NZ from HBLE',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ from HBLE',
       'Remove TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_HFCS_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_N2O_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_OTHER_FCS_STRATEGY_NZ

In [117]:
filtered_emissions_df.head()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
2808,6004.0,700070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.241921,2019,ch4,0.0,0.0,HBLE,UGA,uganda,SISEPUEDE
2809,6004.0,700070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.244762,2020,ch4,0.0,0.0,HBLE,UGA,uganda,SISEPUEDE
2810,6004.0,700070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252282,2021,ch4,0.0,0.0,HBLE,UGA,uganda,SISEPUEDE
2811,6004.0,700070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.251119,2022,ch4,0.0,0.0,HBLE,UGA,uganda,SISEPUEDE
2812,6004.0,700070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252083,2023,ch4,0.0,0.0,HBLE,UGA,uganda,SISEPUEDE


In [118]:
filtered_emissions_df.tail()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
171283,6617.0,1930193.0,Wetlands:co2,Wetlands,LULUCF,0.0,2066,co2,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from ...,UGA,uganda,SISEPUEDE
171284,6617.0,1930193.0,Wetlands:co2,Wetlands,LULUCF,0.0,2067,co2,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from ...,UGA,uganda,SISEPUEDE
171285,6617.0,1930193.0,Wetlands:co2,Wetlands,LULUCF,0.0,2068,co2,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from ...,UGA,uganda,SISEPUEDE
171286,6617.0,1930193.0,Wetlands:co2,Wetlands,LULUCF,0.0,2069,co2,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from ...,UGA,uganda,SISEPUEDE
171287,6617.0,1930193.0,Wetlands:co2,Wetlands,LULUCF,0.0,2070,co2,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from ...,UGA,uganda,SISEPUEDE


In [119]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = filtered_emissions_df
tornado_emissions_df['strategy'].unique()

array(['HBLE', 'Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ from HBLE',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_NZ from HBLE',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_NZ from HBLE',
       'Remove TX:FRST:INCREASE_SEQUESTRATION_NZ from HBLE',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ from HBLE',
       'Remove TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_HFCS_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_N2O_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_OTHER_FCS_STRATEGY_NZ

In [120]:
tornado_emissions_df.head()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
2808,6004.0,700070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.241921,2019,ch4,0.0,0.0,HBLE,UGA,uganda,SISEPUEDE
2809,6004.0,700070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.244762,2020,ch4,0.0,0.0,HBLE,UGA,uganda,SISEPUEDE
2810,6004.0,700070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252282,2021,ch4,0.0,0.0,HBLE,UGA,uganda,SISEPUEDE
2811,6004.0,700070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.251119,2022,ch4,0.0,0.0,HBLE,UGA,uganda,SISEPUEDE
2812,6004.0,700070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252083,2023,ch4,0.0,0.0,HBLE,UGA,uganda,SISEPUEDE


In [121]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,value
0,6004.0,700070.0,HBLE,4901.538826
1,6559.0,1350135.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE,4903.261177
2,6560.0,1360136.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4994.242403
3,6561.0,1370137.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4899.663997
4,6562.0,1380138.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5186.463871


In [122]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,value
55,6613.0,1890189.0,Remove TX:WASO:INC_CAPTURE_BIOGAS_STRATEGY_NZ ...,5086.195151
56,6614.0,1900190.0,Remove TX:WASO:INC_ENERGY_FROM_BIOGAS_STRATEGY...,4899.448463
57,6615.0,1910191.0,Remove TX:WASO:INC_ENERGY_FROM_INCINERATION_ST...,4902.594597
58,6616.0,1920192.0,Remove TX:WASO:INC_LANDFILLING_STRATEGY_NZ fro...,4903.590804
59,6617.0,1930193.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from ...,4923.610319


In [123]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

60
60


In [124]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total column by setting it to the strategy_id == 0 value
base_emission_total = (tornado_emissions_agg_df.loc[tornado_emissions_agg_df['strategy_id'] == 6004, 'emission_total'].values[0])
tornado_emissions_agg_df['base_emission_total'] = base_emission_total 

base_emission_total

np.float64(4901.5388259237225)

In [125]:
# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] =  tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']
tornado_emissions_agg_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff
0,6004.0,700070.0,HBLE,4901.538826,4901.538826,0.000000
1,6559.0,1350135.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE,4903.261177,4901.538826,1.722351
2,6560.0,1360136.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4994.242403,4901.538826,92.703577
3,6561.0,1370137.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4899.663997,4901.538826,-1.874829
4,6562.0,1380138.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5186.463871,4901.538826,284.925045


In [126]:
tornado_emissions_agg_extended_df = add_sector_and_transformation_fields(tornado_emissions_agg_df)
tornado_emissions_agg_extended_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
0,6004.0,700070.0,HBLE,4901.538826,4901.538826,0.000000,NaN,
1,6559.0,1350135.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE,4903.261177,4901.538826,1.722351,AGRC,DEC_CH4_RICE_STRATEGY_NZ
2,6560.0,1360136.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4994.242403,4901.538826,92.703577,AGRC,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ
3,6561.0,1370137.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4899.663997,4901.538826,-1.874829,AGRC,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ
4,6562.0,1380138.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5186.463871,4901.538826,284.925045,AGRC,INC_PRODUCTIVITY_STRATEGY_NZ


In [127]:
tornado_emissions_agg_extended_df.to_clipboard(index=False)

## Load and process CB data

In [128]:
# cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-29T15;28;40.322709_tornado_raw.csv"))
cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "cba_results_ssp_modeling_whirlpool.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value
0,PFLO:HBLE,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
1,PFLO:HBLE,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
2,PFLO:HBLE,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
3,PFLO:HBLE,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
4,PFLO:HBLE,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0


In [129]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

# Split 'variable' into components: name, sector, cb_type, item_1, item_2
# (Assumes exactly 5 colon-separated parts; if there are more colons inside the last field,
# they will be kept in item_2 thanks to n=4)
cb_chars = cb_data["variable"].astype(str).str.split(":", n=4, expand=True)
cb_chars.columns = ["name", "sector", "cb_type", "item_1", "item_2"]
cb_data = pd.concat([cb_data, cb_chars], axis=1)
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2
0,PFLO:HBLE,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
1,PFLO:HBLE,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
2,PFLO:HBLE,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
3,PFLO:HBLE,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
4,PFLO:HBLE,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural


In [130]:
# Scale value from USD to billions (divide by 1e9)
if "value" in cb_data.columns:
    cb_data["value"] = cb_data["value"] / 1e9

# --- Remove "shifted" entries ---
# # Remove rows where item_2 contains "shifted"
# cb_data = cb_data[~cb_data["item_2"].astype(str).str.contains("shifted", na=False)]

# # Remove any remaining rows where variable contains "shifted2"
# cb_data = cb_data[~cb_data["variable"].astype(str).str.contains("shifted2", na=False)]

# --- Add Year column (Year = time_period + 2015) ---
cb_data["Year"] = cb_data["time_period"] + 2015

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year
0,PFLO:HBLE,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019
1,PFLO:HBLE,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020
2,PFLO:HBLE,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021
3,PFLO:HBLE,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022
4,PFLO:HBLE,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023


In [131]:
# Load attribute strategy
attribute_strategy_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "ATTRIBUTE_STRATEGY.csv"))
attribute_strategy_df = attribute_strategy_df[["strategy_id", "strategy_code"]]
attribute_strategy_df.head()

,strategy_id,strategy_code
0,0,BASE
1,6004,PFLO:HBLE
2,6559,WHIRLPOOL_PFLO_HBLE:TX:AGRC:DEC_CH4_RICE_STRAT...
3,6560,WHIRLPOOL_PFLO_HBLE:TX:AGRC:DEC_LOSSES_SUPPLY_...
4,6561,WHIRLPOOL_PFLO_HBLE:TX:AGRC:INC_CONSERVATION_A...


In [132]:
attribute_strategy_df.strategy_id.unique()

array([   0, 6004, 6559, 6560, 6561, 6562, 6563, 6564, 6565, 6566, 6567,
       6568, 6569, 6570, 6571, 6572, 6573, 6574, 6575, 6576, 6577, 6578,
       6579, 6580, 6581, 6582, 6583, 6584, 6585, 6586, 6587, 6588, 6589,
       6590, 6591, 6592, 6593, 6594, 6595, 6596, 6597, 6598, 6599, 6600,
       6601, 6602, 6603, 6604, 6605, 6606, 6607, 6608, 6609, 6610, 6611,
       6612, 6613, 6614, 6615, 6616, 6617])

In [133]:
attribute_strategy_df.strategy_code.unique()

array(['BASE', 'PFLO:HBLE',
       'WHIRLPOOL_PFLO_HBLE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:ENTC:DEC_LOSSES_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:FGTV:DEC_LEAKS_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:FGTV:INC_FLARE_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:FRST:INCREASE_SEQUESTRATION_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:IPPU:DEC_CLINKER_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:IPPU:DEC_HFCS_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:IPPU:DEC_N

In [134]:
# Merge with cb_data on strategy_code
cb_data = cb_data.merge(attribute_strategy_df, on="strategy_code", how="left")
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:HBLE,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019,6004
1,PFLO:HBLE,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020,6004
2,PFLO:HBLE,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021,6004
3,PFLO:HBLE,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022,6004
4,PFLO:HBLE,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023,6004


In [135]:
cb_data.strategy_code.unique()

array(['PFLO:HBLE',
       'WHIRLPOOL_PFLO_HBLE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:ENTC:DEC_LOSSES_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:FGTV:DEC_LEAKS_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:FGTV:INC_FLARE_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:FRST:INCREASE_SEQUESTRATION_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:IPPU:DEC_CLINKER_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:IPPU:DEC_HFCS_STRATEGY_NZ',
       'WHIRLPOOL_PFLO_HBLE:TX:IPPU:DEC_N2O_STRAT

In [136]:
cb_data.strategy_id.unique()

array([6004, 6559, 6560, 6561, 6562, 6563, 6564, 6565, 6566, 6567, 6568,
       6569, 6570, 6571, 6572, 6573, 6574, 6575, 6576, 6577, 6578, 6579,
       6580, 6581, 6582, 6583, 6584, 6585, 6586, 6587, 6588, 6589, 6590,
       6591, 6592, 6593, 6594, 6595, 6596, 6597, 6598, 6599, 6600, 6601,
       6602, 6603, 6604, 6605, 6606, 6607, 6608, 6609, 6610, 6611, 6612,
       6613, 6614, 6615, 6616, 6617])

In [137]:
# check for nans in strategy_id
cb_data[cb_data['strategy_id'].isna()]['strategy_code'].unique()

array([], dtype=object)

In [138]:
cb_data["sector"].unique()

array(['wali', 'entc', 'trns', 'lndu', 'waso', 'trww', 'lvst', 'agrc',
       'ccsq', 'inen', 'scoe', 'ippu', 'soil', 'lsmm', 'fgtv', 'pflo'],
      dtype=object)

In [139]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:HBLE,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019,6004
1,PFLO:HBLE,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020,6004
2,PFLO:HBLE,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021,6004
3,PFLO:HBLE,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022,6004
4,PFLO:HBLE,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023,6004


In [140]:
# filter sectors
# target_sectors = ["wali", "trww", "waso", "soil", "ippu", "lvst", "agrc", "lndu", "lsmm"]
# cb_data = cb_data[cb_data["sector"].isin(target_sectors)].copy()
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:HBLE,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019,6004
1,PFLO:HBLE,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020,6004
2,PFLO:HBLE,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021,6004
3,PFLO:HBLE,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022,6004
4,PFLO:HBLE,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023,6004


In [141]:
# aggregate sum(value) grouped by strategy_id and cb_type
cb_data = (
    cb_data.groupby(["strategy_id", "cb_type"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head(20)

,strategy_id,cb_type,cumulative
0,6004,air_pollution,26.625151
1,6004,congestion,20.962109
2,6004,consumer_savings,645.328072
3,6004,crop_value,74.746558
4,6004,ecosystem_services,21.386132
5,6004,env_pollution,99.385001
6,6004,fuel_cost,145.671458
7,6004,human_health,733.968321
8,6004,ippu_value,1.017973
9,6004,land_pollution,0.537227


In [142]:
# unique cb_data types
cb_cats = cb_data["cb_type"].unique().tolist()

# long -> wide (R dcast equivalent)
wide_cb = (
    cb_data.pivot(index="strategy_id", columns="cb_type", values="cumulative")
      .reset_index()
)

# optional: remove column name from pivot for nicer printing
wide_cb.columns.name = None
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,6004,26.625151,20.962109,645.328072,74.746558,21.386132,99.385001,145.671458,733.968321,1.017973,0.537227,-77.804845,28.616772,95.235397,279.293719,-868.755470,159.455876,187.468079
1,6559,26.625151,20.962109,645.328072,74.746558,21.386132,99.385001,145.671458,733.968321,1.017973,0.537227,-77.804845,28.616772,95.235397,279.293719,-868.599909,159.455876,187.468079
2,6560,26.620185,20.962109,682.695948,141.580738,19.255938,98.512880,149.917212,733.968321,1.017973,0.489859,-78.609346,28.616772,95.230210,279.293719,-892.079956,10.903235,187.468078
3,6561,26.625151,20.962109,606.052311,74.746558,21.386132,99.385001,145.671458,733.968321,1.017973,0.537227,-77.804845,28.616772,95.235397,279.293719,-868.755470,154.855001,187.468079
4,6562,26.661374,20.962109,657.698461,-55.932584,9.867836,99.385001,144.842012,733.968321,1.017973,0.265388,-88.891365,28.616772,95.639963,279.293719,-864.643483,160.875975,187.468080


In [143]:
cb_cats

['air_pollution',
 'congestion',
 'consumer_savings',
 'crop_value',
 'ecosystem_services',
 'env_pollution',
 'fuel_cost',
 'human_health',
 'ippu_value',
 'land_pollution',
 'lvst_value',
 'road_safety',
 'sector_specific',
 'system_cost',
 'technical_cost',
 'technical_savings',
 'water_pollution']

In [144]:
# --- 1) net_benefit = rowSums over all cb categories ---
wide_cb["net_benefit"] = wide_cb[cb_cats].sum(axis=1, skipna=True)

# --- 2) additional_benefits = rowSums excluding "technical_cost" ---
benefit_cols = [c for c in cb_cats if c != "technical_cost"]
wide_cb["additional_benefits"] = wide_cb[benefit_cols].sum(axis=1, skipna=True)

# --- 3) total_transformation_costs = rowSums over specific cols ---
cost_cols = ["technical_cost", "technical_savings", "fuel_cost"]

# (safe version: only use cols that exist in the df)
cost_cols = [c for c in cost_cols if c in wide_cb.columns]

wide_cb["total_transformation_costs"] = wide_cb[cost_cols].sum(axis=1, skipna=True)
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6004,26.625151,20.962109,645.328072,74.746558,21.386132,99.385001,145.671458,733.968321,1.017973,...,-77.804845,28.616772,95.235397,279.293719,-868.755470,159.455876,187.468079,1573.137529,2441.892999,-563.628136
1,6559,26.625151,20.962109,645.328072,74.746558,21.386132,99.385001,145.671458,733.968321,1.017973,...,-77.804845,28.616772,95.235397,279.293719,-868.599909,159.455876,187.468079,1573.293091,2441.892999,-563.472575
2,6560,26.620185,20.962109,682.695948,141.580738,19.255938,98.512880,149.917212,733.968321,1.017973,...,-78.609346,28.616772,95.230210,279.293719,-892.079956,10.903235,187.468078,1505.843875,2397.923831,-731.259509
3,6561,26.625151,20.962109,606.052311,74.746558,21.386132,99.385001,145.671458,733.968321,1.017973,...,-77.804845,28.616772,95.235397,279.293719,-868.755470,154.855001,187.468079,1529.260894,2398.016364,-568.229011
4,6562,26.661374,20.962109,657.698461,-55.932584,9.867836,99.385001,144.842012,733.968321,1.017973,...,-88.891365,28.616772,95.639963,279.293719,-864.643483,160.875975,187.468080,1437.095550,2301.739033,-558.925497


## Merge emissions and cb data and save

In [145]:
tornado_emissions_agg_extended_df[tornado_emissions_agg_extended_df.strategy == "NZ"]

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name


In [146]:
wide_cb.strategy_id.unique()

array([6004, 6559, 6560, 6561, 6562, 6563, 6564, 6565, 6566, 6567, 6568,
       6569, 6570, 6571, 6572, 6573, 6574, 6575, 6576, 6577, 6578, 6579,
       6580, 6581, 6582, 6583, 6584, 6585, 6586, 6587, 6588, 6589, 6590,
       6591, 6592, 6593, 6594, 6595, 6596, 6597, 6598, 6599, 6600, 6601,
       6602, 6603, 6604, 6605, 6606, 6607, 6608, 6609, 6610, 6611, 6612,
       6613, 6614, 6615, 6616, 6617])

In [147]:
print(wide_cb.shape)
print(tornado_emissions_agg_extended_df.shape)

(60, 21)
(60, 8)


In [148]:
df_merged = pd.merge(
    tornado_emissions_agg_extended_df,
    wide_cb,
    on="strategy_id",
    how="inner"
)

df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6004.0,700070.0,HBLE,4901.538826,4901.538826,0.000000,NaN,,26.625151,20.962109,...,-77.804845,28.616772,95.235397,279.293719,-868.755470,159.455876,187.468079,1573.137529,2441.892999,-563.628136
1,6559.0,1350135.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE,4903.261177,4901.538826,1.722351,AGRC,DEC_CH4_RICE_STRATEGY_NZ,26.625151,20.962109,...,-77.804845,28.616772,95.235397,279.293719,-868.599909,159.455876,187.468079,1573.293091,2441.892999,-563.472575
2,6560.0,1360136.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4994.242403,4901.538826,92.703577,AGRC,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,26.620185,20.962109,...,-78.609346,28.616772,95.230210,279.293719,-892.079956,10.903235,187.468078,1505.843875,2397.923831,-731.259509
3,6561.0,1370137.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4899.663997,4901.538826,-1.874829,AGRC,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,26.625151,20.962109,...,-77.804845,28.616772,95.235397,279.293719,-868.755470,154.855001,187.468079,1529.260894,2398.016364,-568.229011
4,6562.0,1380138.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5186.463871,4901.538826,284.925045,AGRC,INC_PRODUCTIVITY_STRATEGY_NZ,26.661374,20.962109,...,-88.891365,28.616772,95.639963,279.293719,-864.643483,160.875975,187.468080,1437.095550,2301.739033,-558.925497


In [149]:
df_merged = df_merged[df_merged['strategy'] != 'HBLE']

df_merged.head()


,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
1,6559.0,1350135.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE,4903.261177,4901.538826,1.722351,AGRC,DEC_CH4_RICE_STRATEGY_NZ,26.625151,20.962109,...,-77.804845,28.616772,95.235397,279.293719,-868.599909,159.455876,187.468079,1573.293091,2441.892999,-563.472575
2,6560.0,1360136.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4994.242403,4901.538826,92.703577,AGRC,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,26.620185,20.962109,...,-78.609346,28.616772,95.230210,279.293719,-892.079956,10.903235,187.468078,1505.843875,2397.923831,-731.259509
3,6561.0,1370137.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4899.663997,4901.538826,-1.874829,AGRC,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,26.625151,20.962109,...,-77.804845,28.616772,95.235397,279.293719,-868.755470,154.855001,187.468079,1529.260894,2398.016364,-568.229011
4,6562.0,1380138.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5186.463871,4901.538826,284.925045,AGRC,INC_PRODUCTIVITY_STRATEGY_NZ,26.661374,20.962109,...,-88.891365,28.616772,95.639963,279.293719,-864.643483,160.875975,187.468080,1437.095550,2301.739033,-558.925497
5,6563.0,1390139.0,Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from HBLE,4902.499773,4901.538826,0.960948,ENTC,DEC_LOSSES_STRATEGY_NZ,26.625006,20.962109,...,-77.804845,28.616772,95.235300,279.293719,-869.307054,159.770482,187.468079,1572.900309,2442.207363,-563.865114


In [150]:
print(df_merged.shape)

(59, 28)


### Below we have some hardcoded fixed exclusive of this study case to replace incorrect tranformation names

In [151]:
df_merged.columns

Index(['strategy_id', 'primary_id', 'strategy', 'emission_total',
       'base_emission_total', 'emission_diff', 'sector', 'transformation_name',
       'air_pollution', 'congestion', 'consumer_savings', 'crop_value',
       'ecosystem_services', 'env_pollution', 'fuel_cost', 'human_health',
       'ippu_value', 'land_pollution', 'lvst_value', 'road_safety',
       'sector_specific', 'system_cost', 'technical_cost', 'technical_savings',
       'water_pollution', 'net_benefit', 'additional_benefits',
       'total_transformation_costs'],
      dtype='object')

In [152]:
# Get the base technical cost from wide_cb (strategy_id 6004)
base_technical_cost = (wide_cb[wide_cb['strategy_id'] == 6004]['technical_cost'].values[0])*-1
base_technical_cost

np.float64(868.7554699905073)

In [153]:
# multiply technical_cost by -1 to get positive costs
df_merged['technical_cost'] = df_merged['technical_cost'] * -1

df_merged['technical_cost'] = base_technical_cost - df_merged['technical_cost'] 

# create marginal total abatement cost column
df_merged['marginal_total_abatement_cost_(USD/tCO2e)'] = (df_merged['technical_cost'] / df_merged['emission_diff'])*1000

# If technical_cost is positive then marginal_total_abatement_cost should be positive too.
df_merged["marginal_total_abatement_cost_(USD/tCO2e)"] = df_merged["marginal_total_abatement_cost_(USD/tCO2e)"].abs() * np.sign(df_merged["technical_cost"])


In [154]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e)
1,6559.0,1350135.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE,4903.261177,4901.538826,1.722351,AGRC,DEC_CH4_RICE_STRATEGY_NZ,26.625151,20.962109,...,28.616772,95.235397,279.293719,0.155561,159.455876,187.468079,1573.293091,2441.892999,-563.472575,90.319228
2,6560.0,1360136.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4994.242403,4901.538826,92.703577,AGRC,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,26.620185,20.962109,...,28.616772,95.230210,279.293719,-23.324486,10.903235,187.468078,1505.843875,2397.923831,-731.259509,-251.602872
3,6561.0,1370137.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4899.663997,4901.538826,-1.874829,AGRC,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,26.625151,20.962109,...,28.616772,95.235397,279.293719,0.000000,154.855001,187.468079,1529.260894,2398.016364,-568.229011,0.000000
4,6562.0,1380138.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5186.463871,4901.538826,284.925045,AGRC,INC_PRODUCTIVITY_STRATEGY_NZ,26.661374,20.962109,...,28.616772,95.639963,279.293719,4.111987,160.875975,187.468080,1437.095550,2301.739033,-558.925497,14.431821
5,6563.0,1390139.0,Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from HBLE,4902.499773,4901.538826,0.960948,ENTC,DEC_LOSSES_STRATEGY_NZ,26.625006,20.962109,...,28.616772,95.235300,279.293719,-0.551584,159.770482,187.468079,1572.900309,2442.207363,-563.865114,-573.999959


In [155]:
df_merged["strategy"].unique()

array(['Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ from HBLE',
       'Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ from HBLE',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ from HBLE',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_NZ from HBLE',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_NZ from HBLE',
       'Remove TX:FRST:INCREASE_SEQUESTRATION_NZ from HBLE',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ from HBLE',
       'Remove TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_HFCS_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_N2O_STRATEGY_NZ from HBLE',
       'Remove TX:IPPU:DEC_OTHER_FCS_STRATEGY_NZ from HB

In [156]:
df_merged['transformation_name_sector'] = df_merged['transformation_name'] + " - " + df_merged['sector']    

In [157]:
df_merged.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_whirlpool.csv"), index=False)

### Create a QA version

In [158]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e),transformation_name_sector
1,6559.0,1350135.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from HBLE,4903.261177,4901.538826,1.722351,AGRC,DEC_CH4_RICE_STRATEGY_NZ,26.625151,20.962109,...,95.235397,279.293719,0.155561,159.455876,187.468079,1573.293091,2441.892999,-563.472575,90.319228,DEC_CH4_RICE_STRATEGY_NZ - AGRC
2,6560.0,1360136.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4994.242403,4901.538826,92.703577,AGRC,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,26.620185,20.962109,...,95.230210,279.293719,-23.324486,10.903235,187.468078,1505.843875,2397.923831,-731.259509,-251.602872,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ - AGRC
3,6561.0,1370137.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4899.663997,4901.538826,-1.874829,AGRC,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,26.625151,20.962109,...,95.235397,279.293719,0.000000,154.855001,187.468079,1529.260894,2398.016364,-568.229011,0.000000,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ - AGRC
4,6562.0,1380138.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5186.463871,4901.538826,284.925045,AGRC,INC_PRODUCTIVITY_STRATEGY_NZ,26.661374,20.962109,...,95.639963,279.293719,4.111987,160.875975,187.468080,1437.095550,2301.739033,-558.925497,14.431821,INC_PRODUCTIVITY_STRATEGY_NZ - AGRC
5,6563.0,1390139.0,Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from HBLE,4902.499773,4901.538826,0.960948,ENTC,DEC_LOSSES_STRATEGY_NZ,26.625006,20.962109,...,95.235300,279.293719,-0.551584,159.770482,187.468079,1572.900309,2442.207363,-563.865114,-573.999959,DEC_LOSSES_STRATEGY_NZ - ENTC


In [159]:
df_merged.sector.unique()

array(['AGRC', 'ENTC', 'FGTV', 'FRST', 'INEN', 'IPPU', 'LNDU', 'LSMM',
       'LVST', 'PFLO', 'SCOE', 'SOIL', 'TRDE', 'TRNS', 'TRWW', 'WALI',
       'WASO'], dtype=object)

In [160]:
relevant_fields = [
    "transformation_name",
    "sector",
    "base_emission_total",
    "emission_total",
    "emission_diff",
    "technical_cost",
    "marginal_total_abatement_cost_(USD/tCO2e)"
]

# keep only relevant fields
df_merged_filtered = df_merged[relevant_fields]
df_merged_filtered.head()

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
1,DEC_CH4_RICE_STRATEGY_NZ,AGRC,4901.538826,4903.261177,1.722351,0.155561,90.319228
2,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,AGRC,4901.538826,4994.242403,92.703577,-23.324486,-251.602872
3,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,AGRC,4901.538826,4899.663997,-1.874829,0.000000,0.000000
4,INC_PRODUCTIVITY_STRATEGY_NZ,AGRC,4901.538826,5186.463871,284.925045,4.111987,14.431821
5,DEC_LOSSES_STRATEGY_NZ,ENTC,4901.538826,4902.499773,0.960948,-0.551584,-573.999959


In [161]:
df_merged_filtered

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
1,DEC_CH4_RICE_STRATEGY_NZ,AGRC,4901.538826,4903.261177,1.722351e+00,0.155561,9.031923e+01
2,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,AGRC,4901.538826,4994.242403,9.270358e+01,-23.324486,-2.516029e+02
3,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,AGRC,4901.538826,4899.663997,-1.874829e+00,0.000000,0.000000e+00
4,INC_PRODUCTIVITY_STRATEGY_NZ,AGRC,4901.538826,5186.463871,2.849250e+02,4.111987,1.443182e+01
5,DEC_LOSSES_STRATEGY_NZ,ENTC,4901.538826,4902.499773,9.609476e-01,-0.551584,-5.740000e+02
6,TARGET_CLEAN_HYDROGEN_STRATEGY_NZ,ENTC,4901.538826,4901.538826,0.000000e+00,0.000000,NaN
7,TARGET_RENEWABLE_ELEC_STRATEGY_NZ,ENTC,4901.538826,4997.819521,9.628069e+01,-7.800535,-8.101868e+01
8,DEC_LEAKS_STRATEGY_NZ,FGTV,4901.538826,4910.389075,8.850249e+00,0.074521,8.420182e+00
9,INC_FLARE_STRATEGY_NZ,FGTV,4901.538826,4901.550832,1.200617e-02,-0.013862,-1.154550e+03
10,INCREASE_SEQUESTRATION_NZ,FRST,4901.538826,4981.944477,8.040565e+01,-0.000248,-3.084372e-03


In [162]:
df_merged_filtered.to_clipboard(index=False)

In [163]:
df_merged_filtered.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_for_QA_whirlpool.csv"), index=False)